# 01 — Data Collection : Cours et composition du STOXX Europe 600

**Projet** : Stratégie quantitative momentum sur le STOXX Europe 600  
**Auteur** : ISLEYEN Volkan  
**Date** : Mai 2026

## Objectif

Collecter les données nécessaires au backtesting d'une stratégie momentum sur le STOXX Europe 600 entre 2015 et 2024.

## Étapes

1. Récupérer la composition (proxy) du STOXX Europe 600
2. Télécharger les cours quotidiens de chaque titre via yfinance
3. Récupérer les données de référence : indice STOXX 600, facteur WML 
   académique (Ken French)
4. Sauvegarder l'ensemble dans `data/raw/` pour les notebooks suivants

## Période d'étude

1er janvier 2015 → 31 décembre 2024 (10 ans)

## Sources

- **Composition STOXX 600** : ETF iShares STOXX Europe 600 (proxy) + Wikipedia
- **Cours individuels** : Yahoo Finance (`yfinance`)
- **Facteur momentum académique (WML)** : Ken French Data Library

In [5]:
# Librairie essentielles 
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
import time

# Configuraton
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

# Détection automatique de la racine du projet
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "output" / "figures"

# Créer les dossiers s'ils n'existent pas
for d in [DATA_RAW, DATA_PROCESSED, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

print("Librairies importées")
print(f"Racine du projet : {PROJECT_ROOT}")
print(f"Données brutes : {DATA_RAW}")
print(f"yfinance version : {yf.__version__}")

Librairies importées
Racine du projet : C:\Users\volka\Documents\momentum-strategy-stoxx600
Données brutes : C:\Users\volka\Documents\momentum-strategy-stoxx600\data\raw
yfinance version : 1.4.1


In [6]:
# Période d'étude
START_DATE = "2015-01-01"
END_DATE = "2024-12-31"

print(f"Période d'étude : {START_DATE} → {END_DATE}")
print(f"Soit environ {(pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)).days} jours")

Période d'étude : 2015-01-01 → 2024-12-31
Soit environ 3652 jours


In [7]:
# Chargement de la composition du STOXX Europe 600
constituents = pd.read_csv(DATA_RAW / "stoxx600_constituents.csv")

print(f"Nombre de titres dans la composition : {len(constituents)}")
print(f"\nColonnes : {constituents.columns.tolist()}")
print(f"\nAperçu :")
print(constituents[["ticker_blackrock", "yahoo_ticker", "nom", "bourse"]].head(10).to_string(index=False))

# Liste des tickers Yahoo pour le téléchargement
tickers = constituents["yahoo_ticker"].tolist()
print(f"\nNombre de tickers Yahoo à télécharger : {len(tickers)}")

Nombre de titres dans la composition : 603

Colonnes : ['ticker_blackrock', 'yahoo_ticker', 'nom', 'secteur', 'ponderation_pct', 'pays', 'bourse', 'devise']

Aperçu :
ticker_blackrock yahoo_ticker               nom                         bourse
            ASML      ASML.AS   ASML HOLDING NV             Euronext Amsterdam
            HSBA       HSBA.L HSBC HOLDINGS PLC          London Stock Exchange
             ROP       ROP.SW   ROCHE PS PAR AG             SIX Swiss Exchange
            NOVN      NOVN.SW       NOVARTIS AG             SIX Swiss Exchange
             AZN        AZN.L   ASTRAZENECA PLC          London Stock Exchange
            NESN      NESN.SW         NESTLE SA             SIX Swiss Exchange
           SHELL     SHELL.AS         SHELL PLC             Euronext Amsterdam
             SIE       SIE.DE      SIEMENS N AG                          Xetra
             TTE       TTE.PA     TOTALENERGIES Nyse Euronext - Euronext Paris
             SAN       SAN.MC   BANCO SANTA

In [5]:
# Test du téléchargement sur 10 titres connus
test_tickers = ["ASML.AS", "MC.PA", "SAP.DE", "NESN.SW", "HSBA.L",
                "SIE.DE", "TTE.PA", "AZN.L", "SAN.MC", "ENEL.MI"]

print("Test de téléchargement sur 10 titres...\n")

test_data = yf.download(
    test_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,     # prix ajustés des dividendes/splits
    progress=True
)

# On garde uniquement les prix de clôture ajustés
test_close = test_data["Close"]

print(f"\nShape des données : {test_close.shape}")
print(f"Période récupérée : {test_close.index.min().date()} → {test_close.index.max().date()}")
print(f"\nAperçu des dernières lignes :")
print(test_close.tail().round(2))

print(f"\nNombre de valeurs manquantes par titre :")
print(test_close.isna().sum())

Test de téléchargement sur 10 titres...



[*********************100%***********************]  10 of 10 completed


Shape des données : (2567, 10)
Période récupérée : 2015-01-02 → 2024-12-30

Aperçu des dernières lignes :
Ticker      ASML.AS     AZN.L  ENEL.MI  HSBA.L   MC.PA  NESN.SW  SAN.MC  \
Date                                                                      
2024-12-20   677.08  10252.76     6.23  760.14  603.55    68.61    4.18   
2024-12-23   674.81  10418.71     6.23  764.44  604.61    68.71    4.15   
2024-12-24   679.44  10432.71      NaN  771.33  606.14      NaN    4.14   
2024-12-27   675.89  10456.70     6.27  776.43  610.76    69.02    4.21   
2024-12-30   663.06  10404.72     6.31  781.02  604.70    69.45    4.22   

Ticker      SAP.DE  SIE.DE  TTE.PA  
Date                                
2024-12-20  230.71  181.08   47.60  
2024-12-23  230.32  180.82   47.56  
2024-12-24     NaN     NaN   47.71  
2024-12-27  233.39  181.41   48.39  
2024-12-30  230.28  181.41   48.25  

Nombre de valeurs manquantes par titre :
Ticker
ASML.AS     8
AZN.L      42
ENEL.MI    24
HSBA.L     42
MC.

In [7]:
# Téléchargement de l'ensemble des 603 titres
# yfinance gère le batch, mais on capture les titres sans données

print(f"Téléchargement de {len(tickers)} titres sur {START_DATE} → {END_DATE}")
print("Cela peut prendre quelques minutes")

# Téléchargeen une fois (yfinance parallélise en interne)
raw_data = yf.download(
    tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True,
    threads=True
)

# Extraction des cours de clôture ajustés
prices = raw_data["Close"]

print(f"nShape brut : {prices.shape}")
print(f"Période : {prices.index.min().date()} → {prices.index.max().date()}")



Téléchargement de 603 titres sur 2015-01-01 → 2024-12-31
Cela peut prendre quelques minutes


[***                    6%                       ]  35 of 602 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: JD..L"}}}
[***                    6%                       ]  38 of 602 completed$JD..L: possibly delisted; no timezone found
[******                13%                       ]  80 of 602 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LEGGR.DE"}}}
[*******               15%                       ]  89 of 602 completed$LEGGR.DE: possibly delisted; no timezone found
[*********             19%                       ]  115 of 602 completed$SBNOR.OL: possibly delisted; no price data found  (1d 2015-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1420066800, endDate = 1735599600")
[**********            20%                       ]  118 of 602 completed$AMV0.DE: possibly delisted; no price data found  (1d

nShape brut : (2586, 602)
Période : 2015-01-01 → 2024-12-30


In [8]:
prices.to_csv(DATA_RAW / "prices_raw.csv")
print(f"Prix sauvegardés : {prices.shape}")

NameError: name 'prices' is not defined

In [9]:
# Rechargement des prix sauvegardés 
prices = pd.read_csv(DATA_RAW / "prices_raw.csv", index_col=0, parse_dates=True)
print(f"Prix rechargés : {prices.shape}")
print(f"Période : {prices.index.min().date()} → {prices.index.max().date()}")

Prix rechargés : (2586, 602)
Période : 2015-01-01 → 2024-12-30


In [14]:
# Diagnostic de la qualité des données par titre
n_obs = len(prices)

# Pourcentage de valeurs manquantes par titre
missing_pct = (prices.isna().sum() / n_obs * 100).sort_values(ascending=False)

print(f"Nombre total de jours : {n_obs}")
print(f"Nombre de titres : {prices.shape[1]}\n")

# Titres entièrement vides
empty_tickers = missing_pct[missing_pct == 100].index.tolist()
print(f"Titres 100% vides : {len(empty_tickers)}")
if empty_tickers :
    print(f"  → {empty_tickers}")

# Titres avec beaucoup de données manquantes (> 20%)
hight_missing = missing_pct[(missing_pct > 20) & (missing_pct < 100)]
print(f"\nTitres avec > 20% de données manquantes : {len(hight_missing)}")
if len(hight_missing) > 0:
    print(hight_missing.round(1).to_string())

# Distribution générale
print(f"\n--- Distribution des valeurs manquantes ---")
print(f"Titres avec 0% manquant      : {(missing_pct == 0).sum()}")
print(f"Titres avec 0-5% manquant    : {((missing_pct > 0) & (missing_pct <= 5)).sum()}")
print(f"Titres avec 5-20% manquant   : {((missing_pct > 5) & (missing_pct <= 20)).sum()}")
print(f"Titres avec > 20% manquant   : {(missing_pct > 20).sum()}")


Nombre total de jours : 2586
Nombre de titres : 602

Titres 100% vides : 23
  → ['--.PA', 'QQ..L', 'RR..L', 'KMAR.OL', 'BP..L', 'JD..L', 'SBNOR.OL', 'LEGGR.DE', 'TEMN.SW', 'BT.A.L', 'AV..L', 'OCTV-SDB.ST', 'BA..L', 'TW..L', 'UU..L', 'AMV0.DE', 'CSG.AS', 'SN..L', 'SUNB.L', 'VSURE.ST', 'FDJU.PA', 'NG..L', 'MICC.AS']

Titres avec > 20% de données manquantes : 52
SUNN.SW        98.8
ZAB.WA         98.1
PUIG.MC        93.4
CVC.AS         93.3
GALD.SW        92.5
R3NK.DE        91.2
SYENS.BR       89.6
SDZ.SW         87.9
MANTA.HE       87.9
LTMC.MI        83.6
DSFIR.AS       83.1
URW.PA         83.1
ACLN.SW        78.1
P911.DE        77.8
EXO.AS         76.4
HLN.L          76.0
VAR.OL         72.1
TPRO.MI        71.7
IVG.MI         70.5
DTG.DE         69.8
UMG.AS         67.5
BPT.L          66.4
WISE.L         66.0
ALLFG.AS       63.3
TE.PA          61.6
AG1.DE         61.5
INPST.AS       61.0
SAVE.ST        60.1
ALE.WA         59.2
ENR.DE         58.0
HAG.DE         57.9
MNG.L          49.

In [16]:
# Identifier les tickers avec un double point (problème de format UK)
double_dot_tickers = [t for t in empty_tickers if ".." in t]
print(f"Tickers à double point détectés : {len(double_dot_tickers)}")
print(double_dot_tickers)

# Correction : remplacer ".." par "."
corrected_map = {t: t.replace("..", ".") for t in double_dot_tickers}
print("\nCorrections proposées :")
for old, new in corrected_map.items():
    print(f"  {old}  →  {new}")

# Retélécharger uniquement les tickers corrigés
corrected_tickers = list(corrected_map.values())
print(f"\nRetéléchargement de {len(corrected_tickers)} tickers corrigés...\n")

recovered = yf.download(
    corrected_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True,
    threads=True
)["Close"]

# Diagnostic des titres récupérés
recovered_missing = (recovered.isna().sum() / len(recovered) * 100)
print(f"\n--- Titres récupérés ---")
for ticker in corrected_tickers:
    if ticker in recovered.columns:
        pct = recovered_missing[ticker]
        status = " OK" if pct < 20 else " encore incomplet"
        print(f"  {ticker:12s} : {pct:5.1f}% manquant  {status}")

[                       0%                       ]

Tickers à double point détectés : 10
['QQ..L', 'RR..L', 'BP..L', 'JD..L', 'AV..L', 'BA..L', 'TW..L', 'UU..L', 'SN..L', 'NG..L']

Corrections proposées :
  QQ..L  →  QQ.L
  RR..L  →  RR.L
  BP..L  →  BP.L
  JD..L  →  JD.L
  AV..L  →  AV.L
  BA..L  →  BA.L
  TW..L  →  TW.L
  UU..L  →  UU.L
  SN..L  →  SN.L
  NG..L  →  NG.L

Retéléchargement de 10 tickers corrigés...



[*********************100%***********************]  10 of 10 completed


--- Titres récupérés ---
  QQ.L         :   0.0% manquant   OK
  RR.L         :   0.0% manquant   OK
  BP.L         :   0.0% manquant   OK
  JD.L         :   0.0% manquant   OK
  AV.L         :   0.0% manquant   OK
  BA.L         :   0.0% manquant   OK
  TW.L         :   0.0% manquant   OK
  UU.L         :   0.0% manquant   OK
  SN.L         :   0.0% manquant   OK
  NG.L         :   0.0% manquant   OK


In [17]:
# 1. Retirer les anciennes colonnes à double point (invalides) du DataFrame principal
prices = prices.drop(columns=double_dot_tickers, errors="ignore")
print(f"Après suppression des colonnes à double point : {prices.shape}")

# 2. Ajouter les colonnes récupérées (tickers corrigés)
prices = prices.join(recovered, how="left")
print(f"Après ajout des titres récupérés : {prices.shape}")

# 3. Retirer les autres titres 100% vides (ceux qui ne sont pas récupérables)
still_empty = [t for t in empty_tickers if ".." not in t]
prices = prices.drop(columns=still_empty, errors="ignore")
print(f"Après suppression des titres non récupérables : {prices.shape}")
print(f"\nTitres définitivement écartés (vides, non récupérables) : {len(still_empty)}")
print(f"  → {still_empty}")

Après suppression des colonnes à double point : (2586, 592)
Après ajout des titres récupérés : (2586, 602)
Après suppression des titres non récupérables : (2586, 589)

Titres définitivement écartés (vides, non récupérables) : 13
  → ['--.PA', 'KMAR.OL', 'SBNOR.OL', 'LEGGR.DE', 'TEMN.SW', 'BT.A.L', 'OCTV-SDB.ST', 'AMV0.DE', 'CSG.AS', 'SUNB.L', 'VSURE.ST', 'FDJU.PA', 'MICC.AS']


In [18]:
# Dernier filtrage : retirer les titres avec trop peu de données exploitables
# Seuil : on garde les titres ayant au moins 2 ans de données (≈ 500 jours)
# car le momentum 12 mois nécessite déjà 1 an d'historique pour le premier signal

MIN_OBS = 500  # environ 2 ans de jours de bourse

# Nombre d'observations valides par titre
valid_obs = prices.notna().sum()

# Titres à garder
tickers_to_keep = valid_obs[valid_obs >= MIN_OBS].index.tolist()
tickers_to_drop = valid_obs[valid_obs < MIN_OBS].index.tolist()

print(f"Seuil minimum : {MIN_OBS} observations valides (≈ 2 ans)\n")
print(f"Titres conservés : {len(tickers_to_keep)}")
print(f"Titres retirés (trop peu de données) : {len(tickers_to_drop)}")
if tickers_to_drop:
    print("\nTitres retirés et leur nombre d'observations :")
    for t in tickers_to_drop:
        print(f"  {t:14s} : {valid_obs[t]:4d} obs ({valid_obs[t]/len(prices)*100:.1f}%)")

# Appliquer le filtre
prices_clean = prices[tickers_to_keep].copy()
print(f"\n--- Univers final ---")
print(f"Shape : {prices_clean.shape}")
print(f"Période : {prices_clean.index.min().date()} → {prices_clean.index.max().date()}")

Seuil minimum : 500 observations valides (≈ 2 ans)

Titres conservés : 577
Titres retirés (trop peu de données) : 12

Titres retirés et leur nombre d'observations :
  CVC.AS         :  174 obs (6.7%)
  DSFIR.AS       :  436 obs (16.9%)
  GALD.SW        :  194 obs (7.5%)
  LTMC.MI        :  423 obs (16.4%)
  MANTA.HE       :  313 obs (12.1%)
  PUIG.MC        :  170 obs (6.6%)
  R3NK.DE        :  228 obs (8.8%)
  SDZ.SW         :  312 obs (12.1%)
  SUNN.SW        :   30 obs (1.2%)
  SYENS.BR       :  268 obs (10.4%)
  URW.PA         :  438 obs (16.9%)
  ZAB.WA         :   48 obs (1.9%)

--- Univers final ---
Shape : (2586, 577)
Période : 2015-01-01 → 2024-12-30


In [19]:
# Sauvegarde de l'univers de prix nettoyé
prices_clean.to_csv(DATA_PROCESSED / "prices_clean.csv")
print(f"Prix nettoyés sauvegardés : {prices_clean.shape}")
print(f"Fichier : {DATA_PROCESSED / 'prices_clean.csv'}")

# On met aussi à jour la liste des constituents pour ne garder que les titres retenus
constituents_final = constituents[constituents["yahoo_ticker"].isin(prices_clean.columns)].copy()

# Ajouter les tickers UK corrigés au mapping (ils ont changé de nom : XX..L → XX.L)
# On vérifie la cohérence
print(f"\nConstituents alignés : {len(constituents_final)} / {len(prices_clean.columns)} colonnes de prix")

Prix nettoyés sauvegardés : (2586, 577)
Fichier : C:\Users\volka\Documents\momentum-strategy-stoxx600\data\processed\prices_clean.csv

Constituents alignés : 568 / 577 colonnes de prix


In [20]:
# Récupération du benchmark : indice STOXX Europe 600
# On teste plusieurs tickers possibles pour l'indice / ETF de référence

benchmark_tickers = {
    "^STOXX": "Indice STOXX Europe 600 (direct)",
    "EXSA.DE": "ETF iShares STOXX 600 (proxy)",
}

print("Test des tickers de benchmark disponibles...\n")

benchmark_data = {}
for ticker, desc in benchmark_tickers.items():
    try:
        data = yf.download(ticker, start=START_DATE, end=END_DATE,
                          auto_adjust=True, progress=False)["Close"]
        if len(data) > 0:
            n_valid = data.notna().sum()
            # data peut être une Series ou un DataFrame selon la version
            n_valid = int(n_valid.iloc[0]) if hasattr(n_valid, 'iloc') else int(n_valid)
            print(f"✓ {ticker:12s} : {n_valid} obs — {desc}")
            benchmark_data[ticker] = data
        else:
            print(f"⨯ {ticker:12s} : aucune donnée")
    except Exception as e:
        print(f"⨯ {ticker:12s} : erreur ({str(e)[:40]})")

print(f"\nBenchmarks récupérés : {list(benchmark_data.keys())}")

Test des tickers de benchmark disponibles...

✓ ^STOXX       : 2515 obs — Indice STOXX Europe 600 (direct)
✓ EXSA.DE      : 2540 obs — ETF iShares STOXX 600 (proxy)

Benchmarks récupérés : ['^STOXX', 'EXSA.DE']


In [21]:
# On retient l'ETF EXSA.DE comme benchmark principal (total return, cohérent avec nos prix ajustés)
# et l'indice ^STOXX comme référence secondaire

benchmark = pd.DataFrame()
benchmark["stoxx600_etf"] = benchmark_data["EXSA.DE"].squeeze()
benchmark["stoxx600_index"] = benchmark_data["^STOXX"].squeeze()

# Aperçu
print("Benchmark STOXX 600 :")
print(f"Shape : {benchmark.shape}")
print(f"Période : {benchmark.index.min().date()} → {benchmark.index.max().date()}")
print(f"\nValeurs manquantes :")
print(benchmark.isna().sum())
print(f"\nAperçu :")
print(benchmark.tail().round(2))

# Sauvegarde
benchmark.to_csv(DATA_RAW / "benchmark_stoxx600.csv")
print(f"\nBenchmark sauvegardé : {DATA_RAW / 'benchmark_stoxx600.csv'}")

Benchmark STOXX 600 :
Shape : (2540, 2)
Période : 2015-01-02 → 2024-12-30

Valeurs manquantes :
stoxx600_etf       0
stoxx600_index    28
dtype: int64

Aperçu :
            stoxx600_etf  stoxx600_index
Date                                    
2024-12-19         48.66          506.66
2024-12-20         48.28          502.19
2024-12-23         48.32          502.91
2024-12-27         48.72          507.18
2024-12-30         48.52          504.85

Benchmark sauvegardé : C:\Users\volka\Documents\momentum-strategy-stoxx600\data\raw\benchmark_stoxx600.csv


In [24]:
# Chargement du facteur momentum WML européen (Ken French)
# Corrections : skiprows=7 (ligne ",WML" incluse), date lue en float à reconvertir

wml_raw = pd.read_csv(
    DATA_RAW / "Europe_MOM_Factor_Daily.csv",
    skiprows=7,
    names=["date", "WML"],
    skipinitialspace=True
)

# La date est lue comme float (ex: 19901101.0) → int → str
wml_raw = wml_raw.dropna(subset=["date"])
wml_raw["date"] = wml_raw["date"].astype(float).astype(int).astype(str)

# Garder les dates valides (8 chiffres) et convertir
wml_raw = wml_raw[wml_raw["date"].str.match(r"^\d{8}$")]
wml_raw["date"] = pd.to_datetime(wml_raw["date"], format="%Y%m%d")
wml_raw = wml_raw.set_index("date")

# WML en numérique, -99.99 → NaN
wml_raw["WML"] = pd.to_numeric(wml_raw["WML"], errors="coerce").replace(-99.99, np.nan)

# Filtrer sur la période d'étude
wml = wml_raw[(wml_raw.index >= START_DATE) & (wml_raw.index <= END_DATE)].copy()

print(f"Facteur WML chargé : {wml.shape}")
print(f"Période : {wml.index.min().date()} → {wml.index.max().date()}")
print(f"\nStatistiques du WML (rendement quotidien en %) :")
print(wml["WML"].describe().round(4))
print(f"\nAperçu :")
print(wml.head())

# Sauvegarde
wml.to_csv(DATA_PROCESSED / "wml_factor.csv")
print(f"\nSauvegardé : {DATA_PROCESSED / 'wml_factor.csv'}")

Facteur WML chargé : (2609, 1)
Période : 2015-01-01 → 2024-12-31

Statistiques du WML (rendement quotidien en %) :
count    2609.0000
mean        0.0326
std         0.7401
min       -10.8700
25%        -0.3000
50%         0.0700
75%         0.4100
max         4.5600
Name: WML, dtype: float64

Aperçu :
             WML
date            
2015-01-01  0.00
2015-01-02  0.00
2015-01-05  1.51
2015-01-06 -0.08
2015-01-07  0.05

Sauvegardé : C:\Users\volka\Documents\momentum-strategy-stoxx600\data\processed\wml_factor.csv


## Récapitulatif du notebook 01

### Données collectées

| Source | Contenu | Observations |
|--------|---------|--------------|
| iShares STOXX 600 (BlackRock) | Composition de l'indice | 603 titres |
| Yahoo Finance | Cours quotidiens ajustés | 577 titres retenus, 2015-2024 |
| Yahoo Finance | Benchmark STOXX 600 (ETF + indice) | 2 540 jours |
| Ken French Data Library | Facteur momentum WML européen | 2 609 jours |

### Traitement de la qualité des données

À partir des 603 titres de la composition initiale :
- 602 titres téléchargés avec succès via Yahoo Finance
- 10 titres britanniques récupérés après correction du format de ticker 
  (double point : `BP..L` → `BP.L`)
- 13 titres écartés (tickers introuvables, titres absorbés comme Credit Suisse)
- 12 titres écartés pour historique insuffisant (IPO récentes : Puig, Porsche, etc.)
- **Univers final : 577 titres** avec au moins 2 ans de données

### Limites documentées

La composition utilisée est celle de mai 2026 (composition actuelle de l'ETF). 
L'analyse est donc sujette à un **survivorship bias** : les titres sortis de 
l'indice avant 2026 ne sont pas inclus, ce qui peut surestimer la performance. 
Une version sans biais nécessiterait la composition historique mensuelle de 
l'indice (disponible via Bloomberg).

### Fichiers produits

- `data/processed/prices_clean.csv` — cours quotidiens des 577 titres
- `data/raw/benchmark_stoxx600.csv` — benchmark STOXX 600
- `data/processed/wml_factor.csv` — facteur momentum académique

### Prochaine étape

Notebook 02 : construction du signal momentum (rendements passés, ranking) 
et formation des portefeuilles.